# Camada Bronze - Global Superstore + World Data 2017 (Gabriel)

Carrega os ficheiros originais para o HDFS e cria as tabelas Bronze em formato Delta, na stack comum do grupo.
Os dados ficam tal como vêm da fonte (todas as colunas como texto, com os nomes originais),
com os metadados de ingestão `_data_ingestao` e `_ficheiro_origem`.

| Dataset | Ficheiro | Tabela Bronze |
|---|---|---|
| Global Superstore (principal) | Global Superstore Data.xlsx (folha `Orders`) | bronze.superstore_orders |
| World Data 2017 (complementar) | World-Data 2017-Global-Economic-Environmental-Indicator.csv | bronze.world_data_2017 |

Fontes: Kaggle `rohitgrewal/global-superstore-data` (versão com encomendas de 2014 a 2017) e
`meida18/world-data-2017-global-economic-indicator`.

O Spark não lê Excel, por isso a folha `Orders` é lida com pandas (tudo como texto) e passada para Spark sem
qualquer transformação. O Excel original fica guardado no HDFS. A Bronze guarda todas as datas; o filtro
de junho a dezembro de 2017 é feito na Silver.

In [1]:
XLSX_LOCAL = "/home/jovyan/work/data/gabriel/Global Superstore Data.xlsx"
WORLD_LOCAL = "/home/jovyan/work/data/gabriel/World-Data 2017-Global-Economic-Environmental-Indicator.csv"
XLSX_HDFS = "/projeto/bronze/gabriel/global_superstore_data.xlsx"
WORLD_HDFS = "/projeto/bronze/gabriel/world_data_2017.csv"

ficheiros = {XLSX_LOCAL: XLSX_HDFS, WORLD_LOCAL: WORLD_HDFS}

In [2]:
# Copiar os ficheiros originais para a zona Bronze do HDFS (full load: apaga a versão anterior e volta a carregar)
from hdfs import InsecureClient
client = InsecureClient("http://hdfs-nn:9870", user="anonymous")

for from_path, to_path in ficheiros.items():
    client.delete(to_path)
    client.upload(to_path, from_path)
    print(to_path, client.status(to_path)["length"], "bytes")

/projeto/bronze/gabriel/global_superstore_data.xlsx 7296183 bytes
/projeto/bronze/gabriel/world_data_2017.csv 28708 bytes


In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window
from delta import *

# Stack comum do grupo: HDFS (hdfs-nn), Hive Metastore e Delta Lake 2.4
warehouse_location = "hdfs://hdfs-nn:9000/warehouse"

builder = SparkSession \
    .builder \
    .appName("Bronze - Global Superstore + World Data (Gabriel)") \
    .config("spark.driver.memory", "2g") \
    .config("spark.sql.warehouse.dir", warehouse_location) \
    .config("hive.metastore.uris", "thrift://hive-metastore:9083") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0") \
    .config("spark.sql.session.timeZone", "UTC") \
    .enableHiveSupport()

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# Os nomes originais das colunas têm espaços e caracteres especiais;
# o column mapping do Delta permite guardá-los tal como vêm da fonte
spark.conf.set("spark.databricks.delta.properties.defaults.columnMapping.mode", "name")
spark.conf.set("spark.databricks.delta.properties.defaults.minReaderVersion", "2")
spark.conf.set("spark.databricks.delta.properties.defaults.minWriterVersion", "5")

In [4]:
spark.sql("CREATE DATABASE IF NOT EXISTS bronze LOCATION 'hdfs://hdfs-nn:9000/warehouse/bronze.db/'")

def gravar_bronze(df, tabela):
    """Acrescenta os metadados de ingestão e grava a tabela Delta tal como veio da fonte."""
    df = df.withColumn("_data_ingestao", F.current_timestamp()) \
           .withColumn("_ficheiro_origem", F.input_file_name())
    df.write \
        .mode("overwrite") \
        .format("delta") \
        .option("overwriteSchema", "true") \
        .option("path", f"hdfs://hdfs-nn:9000/warehouse/bronze.db/{tabela}/") \
        .saveAsTable(f"bronze.{tabela}")
    print(f"bronze.{tabela}: {spark.table(f'bronze.{tabela}').count()} linhas, {len(df.columns) - 2} colunas originais")

def ler_csv(hdfs_path, **opcoes):
    """Lê um CSV do HDFS com todas as colunas como texto (sem inferSchema)."""
    leitor = spark.read.option("header", "true").option("multiLine", "true").option("escape", '"')
    for k, v in opcoes.items():
        leitor = leitor.option(k, v)
    return leitor.csv("hdfs://hdfs-nn:9000" + hdfs_path)

In [5]:
import pandas as pd

# Global Superstore: folha Orders, todas as colunas como texto e sem converter vazios em NaN
orders_pd = pd.read_excel(XLSX_LOCAL, sheet_name="Orders", dtype=str, keep_default_na=False, engine="openpyxl")
print("Folha Orders:", orders_pd.shape)
orders = spark.createDataFrame(orders_pd.astype(str))

# O input_file_name() só funciona com ficheiros lidos pelo Spark; aqui a origem é o Excel no HDFS
orders = orders.withColumn("_data_ingestao", F.current_timestamp()) \
               .withColumn("_ficheiro_origem", F.lit("hdfs://hdfs-nn:9000" + XLSX_HDFS + "#Orders"))
orders.write.mode("overwrite").format("delta").option("overwriteSchema", "true") \
    .option("path", "hdfs://hdfs-nn:9000/warehouse/bronze.db/superstore_orders/") \
    .saveAsTable("bronze.superstore_orders")
print("bronze.superstore_orders:", spark.table("bronze.superstore_orders").count(), "linhas,", len(orders_pd.columns), "colunas originais")

# World Data 2017. O ficheiro tem fins de linha Windows (\r\n): a última coluna do cabeçalho chegaria como "Year\r".
# Só o nome da coluna é corrigido; os valores ficam tal como vêm da fonte (o "\r" dos valores é limpo na Silver).
world = ler_csv(WORLD_HDFS)
world = world.toDF(*[c.rstrip("\r") for c in world.columns])
gravar_bronze(world, "world_data_2017")

Folha Orders: (51290, 24)
bronze.superstore_orders: 51290 linhas, 24 colunas originais
bronze.world_data_2017: 195 linhas, 36 colunas originais


## Verificação

As tabelas Bronze têm de ter o mesmo número de linhas dos ficheiros originais.

In [6]:
import csv
with open(WORLD_LOCAL, encoding="utf-8-sig", newline="") as f:
    n_world = sum(1 for _ in csv.reader(f)) - 1
for tabela, n in {"superstore_orders": len(orders_pd), "world_data_2017": n_world}.items():
    real = spark.table(f"bronze.{tabela}").count()
    print(f"{tabela:18s} ficheiro {n:>6}  bronze {real:>6}  {'OK' if n == real else 'DIFERENTE'}")
    assert n == real

superstore_orders  ficheiro  51290  bronze  51290  OK
world_data_2017    ficheiro    195  bronze    195  OK


In [7]:
spark.table("bronze.superstore_orders").select("Row ID", "Order ID", "Order Date", "Country", "Market", "Sales", "Discount", "Profit").show(3, truncate=False)
spark.table("bronze.world_data_2017").select("Country", "Abbreviation", "GDP", "Unemployment rate", "Year").show(3, truncate=False)

+------+-----------------------+-------------------+--------------------+------------+------+--------+--------+
|Row ID|Order ID               |Order Date         |Country             |Market      |Sales |Discount|Profit  |
+------+-----------------------+-------------------+--------------------+------------+------+--------+--------+
|43924 |AE-2017-GH4665138-43082|2017-12-13 00:00:00|United Arab Emirates|Asia Pacific|59.373|0.7     |-118.767|
|43922 |AE-2017-GH4665138-43082|2017-12-13 00:00:00|United Arab Emirates|Asia Pacific|77.256|0.7     |-82.584 |
|43925 |AE-2017-GH4665138-43082|2017-12-13 00:00:00|United Arab Emirates|Asia Pacific|42.489|0.7     |-62.331 |
+------+-----------------------+-------------------+--------------------+------------+------+--------+--------+
only showing top 3 rows

+-----------+------------+------------+-----------------+------+
|Country    |Abbreviation|GDP         |Unemployment rate|Year  |
+-----------+------------+------------+-----------------+----

In [8]:
spark.stop()